# Imports

In [ ]:
!pip install torch_geometric

In [ ]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import math
import gc

import torch
import torch.nn as nn

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

from datetime import datetime, timezone, timedelta

from sklearn.metrics import (precision_score, recall_score, f1_score, fbeta_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve)
from sklearn.calibration import calibration_curve

import random
import json
import matplotlib.pyplot as plt
import time

In [ ]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [ ]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir

        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self,
                       model_config=None,
                       model_name=None,
                       params=None,
                       metrics=None,
                       model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)
        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        tz = timezone(timedelta(hours=-3)) #Argentina
        now = datetime.now(tz)

        # -------------------------
        # Extract config
        # -------------------------
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = params

        run_ts = extra_params.get("run_ts", None)
        run_id = extra_params.get("run_id", None)

        # -------------------------
        # Canonical timestamp
        # -------------------------
        if run_ts is not None:
            run_dt = datetime.strptime(run_ts, "%Y%m%d_%H%M%S").replace(tzinfo=tz)
        else:
            run_dt = now
            run_ts = run_dt.strftime("%Y%m%d_%H%M%S")

        # -------------------------
        # Create CSV entry
        # -------------------------
        entry = {
            "timestamp": run_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "run_ts": run_ts,
            "run_id": run_id,
            "model_name": mname
        }

        if mtype is not None:
            entry["type"] = mtype

        if threshold is not None:
            entry["prob_threshold"] = threshold

        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        df_new = pd.DataFrame([entry])

        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"\nExperiment recorded in {self.log_file}")

        # -------------------------
        # Save model
        # -------------------------
        if model_object is not None:

            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"

            # base_model_name = mtype if mtype is not None else mname
            # model_sub_dir = os.path.join(self.model_dir, base_model_name)
            #os.makedirs(model_sub_dir, exist_ok=True)
            os.makedirs(self.model_dir, exist_ok=True)

            if run_id:
                filename = f"{run_id}_{safe_key}_{float(metric_val):.4f}"
            else:
                filename = f"{mname}_{run_ts}_{safe_key}_{float(metric_val):.4f}"

            #filepath = os.path.join(model_sub_dir, filename)
            filepath = os.path.join(self.model_dir, filename)

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, f"{filepath}.joblib")
            else:
                import torch
                torch.save(model_object.state_dict(), f"{filepath}.pth")

            print(f"Saved model: {filepath}")


# Load data

In [ ]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0001, mode='max'):
        """
        Stop training if the metric doesn't improve after 'patience' epochs
        mode='max': For metrics that need to increase (e.g., AUC, F1)
        mode='min': For metrics that need to decrease (e.g., Loss)
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.early_stop = False
        self.best_score = -np.inf if mode == 'max' else np.inf
        self.best_model_state = None
        self.best_epoch = None

    def __call__(self, current_score, model, epoch=None):
        if self.mode == 'max':
            improved = current_score > (self.best_score + self.min_delta)
        else:
            improved = current_score < (self.best_score - self.min_delta)

        if improved:
            self.best_score = current_score
            self.best_model_state = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            self.counter = 0
            return True # Indicates that there was an improvement
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False


# Model

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super().__init__()

        # Input: Only flow attributes (edge_attr)
        # The MLP only needs to know how many features the edge has.
        self.input_dim = edge_dim

        # Note: node_dim is received for compatibility but is not used

        # We use LayerNorm instead of BatchNorm1d
        # LayerNorm does not fail with batch_size=1
        self.net = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.net[-1].bias.data.fill_(output_bias_init)

    def forward(self, x, edge_index, edge_attr):
        """
        Arguments:
            x, edge_index: These are received to maintain compatibility with
                           train_epoch, but this model ignores them (it doesn't
                           use a graph structure).
            edge_attr: The flow's features (bytes, duration, etc.)
        """
        # We ignore everything except the attributes of the flow
        return self.net(edge_attr)


# Functions


## Metrics

In [ ]:
def calculate_metrics_gnn(y_true, y_probs, prob_threshold=0.5):
    """
    y_true: Array of actual labels (0 or 1)
    y_probs: Array de PROBABILITIES (0.0 to 1.0). NO LOGITS.
    """
    y_true = np.array(y_true)
    probs = np.array(y_probs)

    # Predictions (0 or 1) using the threshold
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0; roc = 0.5

    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2,
        "AUC-PR": ap, "AUC-ROC": roc, "FPR": fpr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

## Train and eval

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0



In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, is_temporal=False):
    """
    Just run the model and return Loss and raw Probabilities
    """
    model.eval()
    if is_temporal and hasattr(model, 'reset_memory'): model.reset_memory()

    all_probs = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)
        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # 1. Loss (using logits)
        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        # 2. Probs (using sigmoid)
        probs = torch.sigmoid(out.view(-1))
        all_probs.extend(probs.cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    avg_loss = total_loss / steps if steps > 0 else 0.0

    # Only return the raw data
    return avg_loss, np.array(all_trues), np.array(all_probs)

## Save plots

In [ ]:
def save_plots(train_loss, val_loss, y_true, y_probs, seed, exp_name, run_ts, save_dir="./plots"):
    os.makedirs(save_dir, exist_ok=True)

    # 1. Loss Curve
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, label='Train Loss')
    plt.plot(val_loss, label='Val Loss', linestyle='--')
    plt.title(f'{exp_name} - Loss Seed {seed}')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{save_dir}/loss_{exp_name}_seed{seed}_{run_ts}.png")
    plt.close()

    # 2. Calibration Curve
    prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=10)
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated')
    plt.plot(prob_pred, prob_true, marker='.', label='Model')
    plt.title(f'{exp_name} - Calibration Seed {seed}')
    plt.xlabel('Mean Predicted Probability'); plt.ylabel('Fraction of Positives')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(f"{save_dir}/calib_{exp_name}_seed{seed}_{run_ts}.png")
    plt.close()

## Optimal threshold

In [ ]:
def find_optimal_threshold_constrained(model, loader, device, is_temporal=False, min_precision=0.90):
    """
    Find the threshold that maximizes the recall rate subject to a precision constraint.
    If the constraint is not met, fallback to max F1.
    """
    model.eval()
    if is_temporal and hasattr(model, 'reset_memory'): model.reset_memory()

    all_probs, all_trues = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            if data.x.shape[0] == 0: continue

            # Forward
            if is_temporal:
                out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
            else:
                out = model(data.x, data.edge_index, data.edge_attr)

            # Convert Logits to Probabilities (Sigmoid)
            probs = torch.sigmoid(out.view(-1))
            all_probs.extend(probs.cpu().numpy())
            all_trues.extend(data.y.cpu().numpy())

    y_true = np.array(all_trues)
    y_probs = np.array(all_probs)

    # Precision-Recall Curve
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    precisions, recalls = precisions[:-1], recalls[:-1] # Adjust dimensions

    # Strategy: Max Recall subject to Precision > min_precision
    valid_indices = np.where(precisions >= min_precision)[0]

    if len(valid_indices) > 0:
        best_idx = valid_indices[np.argmax(recalls[valid_indices])]
        strategy = f"Max Recall @ Prec>={min_precision}"
    else:
        # Fallback: Max F1
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        strategy = "Max F1 (Fallback)"

    best_th = thresholds[best_idx]
    print(f"   \nOptimal Threshold: {best_th:.4f} ({strategy})")

    # Return complete vectors to graph calibration afterwards
    return best_th, y_true, y_probs

## Run multiple seeds

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
def run_multiple_seeds(model_class, model_config, train_loader, val_loader,
                       manager,
                       seeds=[42, 123, 777, 2024, 99],
                       epochs=60,
                       device='cpu',
                       experiment_name="Exp_Optimized",
                       json_dir="./logs",
                       plots_dir="./plots"):

    os.makedirs(json_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    print(f" Starting Multi-Seed Run: {experiment_name}")
    print(f"   Seeds: {seeds}")
    print("-" * 60)

    for seed in seeds:
        t0_seed = time.perf_counter()
        t_train_total = 0.0
        t_eval_total = 0.0
        t_threshold_total = 0.0

        tz = timezone(timedelta(hours=-3))  # Argentina
        run_ts = datetime.now(tz).strftime("%Y%m%d_%H%M%S")
        run_id = f"{experiment_name}_seed{seed}_{run_ts}"

        print(f"\nRunning seed {seed} | run_id={run_id}")

        # Name
        exp_id = f"{experiment_name}_seed{seed}"
        print(f"\n{exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        set_seed(seed)

        # Update model_config
        model_config['model_name'] = exp_id
        model_config.setdefault("extra_params", {})
        model_config["extra_params"]["run_ts"] = run_ts
        model_config["extra_params"]["run_id"] = run_id

        # Instantiate Model
        model = model_class(**model_config['model_params']).to(device)

        # Training Setup
        optimizer = torch.optim.Adam(model.parameters(), lr=model_config['extra_params']['learning_rate'])
        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # Early Stopping
        early_stopping = EarlyStopping(patience=10, mode='max', min_delta=0.0001)

        train_hist, val_loss_hist, val_auc_hist = [], [], []

        # Epochs
        for epoch in range(epochs):
            # Detect whether it is a temporal model (ST or GRU) to adjust inputs
            is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

            # --- TRAIN ---
            t0 = time.perf_counter()
            loss_train = train_epoch(model, train_loader, optimizer, criterion,
                                     device=device, is_temporal=is_temporal,
                                     batch_steps=model_config['extra_params']["batch_steps"])
            t_train_total += time.perf_counter() - t0

            # --- EVAL ---
            t0 = time.perf_counter()
            val_loss, y_true, y_probs = evaluate(model, val_loader, criterion, device, is_temporal=is_temporal)
            t_eval_total += time.perf_counter() - t0

            # Monitor: calculate only AUC-PR
            val_auc = average_precision_score(y_true, y_probs)

            # Save history
            train_hist.append(float(loss_train))
            val_loss_hist.append(float(val_loss))
            val_auc_hist.append(float(val_auc))

            # Early Stopping check
            improved = early_stopping(val_auc, model, epoch)

            if improved or (epoch+1) % 10 == 0:
                mark = "(*)" if improved else ""
                print(f"   Ep {epoch+1} | Loss: {loss_train:.4f} | Val Loss: {val_loss:.4f} | Val AUC-PR: {val_auc:.4f} {mark}")

            if early_stopping.early_stop:
                print(f"   Early Stopping in epoch {epoch}")
                break


        # Restoring the best model
        model.load_state_dict(early_stopping.best_model_state)



        # 1. Re-evaluate the BEST model to obtain the clean probs
        _, y_true_best, y_probs_best = evaluate(model, val_loader, criterion, device, is_temporal=is_temporal)

        # 2. We are looking for the optimal threshold (Max Recall subject to Precision)
        t0 = time.perf_counter()
        best_th, _, _ = find_optimal_threshold_constrained(
            model, val_loader, device, is_temporal, min_precision=0.90
        )
        t_threshold_total += time.perf_counter() - t0

        # 3. We calculate ALL complex metrics
        final_metrics = calculate_metrics_gnn(y_true_best, y_probs_best, prob_threshold=best_th)
        final_metrics['optimal_threshold'] = best_th
        final_metrics['stopped_epoch'] = len(train_hist)
        final_metrics['seed'] = seed
        final_metrics["run_ts"] = run_ts
        final_metrics["run_id"] = run_id
        final_metrics["time_total_sec"] = time.perf_counter() - t0_seed
        final_metrics["time_train_sec"] = t_train_total
        final_metrics["time_eval_sec"] = t_eval_total
        final_metrics["time_threshold_sec"] = t_threshold_total



        print(f"   Final: Precision={final_metrics['Precision']:.4f} |  Recall={final_metrics['Recall']:.4f} | F1={final_metrics['F1']:.4f} | F2={final_metrics['F2']:.4f} | AUC-PR={final_metrics['AUC-PR']:.4f} | FPR={final_metrics['FPR']:.5f}")

        # 4. Plots and Manager
        save_plots(train_hist, val_loss_hist, y_true_best, y_probs_best, seed, experiment_name, run_ts, save_dir=os.path.join(plots_dir,experiment_name))

        manager.log_experiment(
            model_config=model_config,
            metrics=final_metrics,
            model_object=model
        )

        # Save JSON
        filename_json = os.path.join(
            json_dir, experiment_name,
            f"training_history_{experiment_name}_seed{seed}_{run_ts}.json"
        )

        history_payload = {
            "experiment_name": experiment_name,
            "seed": seed,
            "run_ts": run_ts,
            "run_id": run_id,
            "training": {
                "train_loss": train_hist,
                "val_loss": val_loss_hist,
                "val_aucpr": val_auc_hist
            },
            "early_stopping": {
                "best_epoch": int(early_stopping.best_epoch),
                "best_val_aucpr": float(early_stopping.best_score),
                "stopped_epoch": int(len(train_hist))
            }
        }


        try:
            with open(filename_json, 'w') as f:
                json.dump(history_payload, f, cls=NumpyEncoder, indent=4)
            print(f"Training history saved in: {filename_json}")
        except Exception as e:
            print(f"\nWarning: JSON could not be saved: {e}")


        print(f"\n End seed {seed}")

        print("-" * 60)

    # 8. Print final statistics
    df = pd.read_csv(manager.log_file)
    df_exp = df[df["model_name"].str.contains(experiment_name)]

    if len(df_exp) == 0:
        print("No records found for this experiment.")
    else:

        def mean_std(series):
            return series.mean(), series.std()

        auc_mean, auc_std = mean_std(df_exp["AUC-PR"])
        rec_mean, rec_std = mean_std(df_exp["Recall"])
        ttot_mean, ttot_std = mean_std(df_exp["time_total_sec"])
        ttrain_mean, ttrain_std = mean_std(df_exp["time_train_sec"])

        print("="*50)
        print(f" AVERAGE RESULT: {experiment_name}")
        print(f"AUC-PR:      {auc_mean:.4f} ± {auc_std:.4f}")
        print(f"Recall:      {rec_mean:.4f} ± {rec_std:.4f}")
        print(f"Total Time:  {ttot_mean:.2f} ± {ttot_std:.2f} sec")
        print(f"Train Time:  {ttrain_mean:.2f} ± {ttrain_std:.2f} sec")
        print("="*50)



# Auxiliary

In [ ]:
ROOT_PATH = "./dataset_processed"

In [ ]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [ ]:
ROOT_DIR = "./results_earlystopping"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")


# Main

## Seeds

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [ ]:
model_config = {
    "model_name": None,
    "type": "SimpleMLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
EXPERIMENT_NAME="SimpleMLP_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [ ]:
run_multiple_seeds(
    model_class=SimpleMLP,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: SimpleMLP_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=SimpleMLP_BiasOn_seed42_20260219_181723

SimpleMLP_BiasOn_seed42
   Ep 1 | Loss: 0.2027 | Val Loss: 0.2297 | Val AUC-PR: 0.0904 (*)
   Ep 4 | Loss: 0.1982 | Val Loss: 0.2255 | Val AUC-PR: 0.1557 (*)
   Ep 5 | Loss: 0.1957 | Val Loss: 0.2244 | Val AUC-PR: 0.1590 (*)
   Ep 6 | Loss: 0.1949 | Val Loss: 0.2235 | Val AUC-PR: 0.1720 (*)
   Ep 7 | Loss: 0.1943 | Val Loss: 0.2230 | Val AUC-PR: 0.1728 (*)
   Ep 8 | Loss: 0.1940 | Val Loss: 0.2224 | Val AUC-PR: 0.1759 (*)
   Ep 10 | Loss: 0.1934 | Val Loss: 0.2221 | Val AUC-PR: 0.1776 (*)
   Ep 17 | Loss: 0.1930 | Val Loss: 0.2211 | Val AUC-PR: 0.1836 (*)
   Ep 20 | Loss: 0.1919 | Val Loss: 0.2210 | Val AUC-PR: 0.1738 
   Early Stopping in epoch 26
   
Optimal Threshold: 0.2123 (Max Recall @ Prec>=0.9)
   Final: Precision=0.9015 |  Recall=0.0190 | F1=0.0372 | F2=0.0236 | AUC-PR=0.183